In [ ]:
!pip install git+https://github.com/facebookresearch/fairchem.git@fairchem_core-2.0.0#subdirectory=packages/fairchem-core

In [16]:
# colab
!git clone https://github.com/yasheshak/Chem-277B-Final-Project.git

Cloning into 'Chem-277B-Final-Project'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 100 (delta 40), reused 79 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 2.15 MiB | 12.02 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [17]:
%cd Chem-277B-Final-Project

/content/Chem-277B-Final-Project/Chem-277B-Final-Project


In [18]:
!ls -a

.			    extract.py		      SchNet_for_import.py
..			    .git		      SchNet_GNN_Baseline.ipynb
db_explore.ipynb	    hyperparam_test.ipynb     SchNet_GNN.ipynb
DimeNet_baseline_new.ipynb  hyperparam_test_v2.ipynb  SchNet_Normalize.ipynb
EDA			    Makefile		      simpleGNN
EDA.ipynb		    README.md
environment.yml		    read_multi_ase.py


Goal: Optimizer hyperparameters

Plan
1. Baseline sweep: identify how sensitive the model is when changing target parameter. A range of values for a single parameter is tested while keeping the other params at its default value.
    - Narrow down hyperparameter values
2. Grid search: explores interactions between hyperparameters by examining every possible combination.

Baseline sweep
1. create dict of start, stop, min (user defined)


In [19]:
import sys
sys.path.append('/content/Chem-277B-Final-Project')
from read_multi_ase import *
from extract import *
from SchNet_for_import import *

import numpy as np
import glob
from typing import Union
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split
from torch_geometric.data import Data
from torch_geometric.nn import SchNet
from torch_geometric.loader import DataLoader
from fairchem.core.datasets import AseDBDataset

In [9]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
# ---------------------------------- Initialize hyperparam range -------------------------------
# parameters to optimize
hidden_channels = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_filters
num_filters = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_interactions
num_interactions = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_gaussians
num_gaussians = {
    'start': 1,
    'end': 1,
    'step': 1
}
# cutoff
cutoff = {
    'start': 1,
    'end': 1,
    'step': 1
}
# max_num_neighbors
max_num_neighbors = {
    'start': 1,
    'end': 1,
    'step': 1
}
params_to_optimize = [hidden_channels, num_filters, num_interactions, num_gaussians, cutoff, max_num_neighbors]

# set parameters
# set_params = [readout, dipole, mean, std, atomref, train_mean, train_std]

In [22]:
# ------------------------------ Initialize training set --------------------------------
# given the local dataset path, loads the first .aselmdb file
dataset_path = '/content/drive/MyDrive/train_4M/data0000.aselmdb'
# dataset = AseDBDataset({"src": dataset_path})
files_list = dataset_path
dataset = process_file(files_list, molecule_type = 'biomolecules', max_molecules = 200)

# convert to torch
torch_data = get_data(dataset)
train_dataset, val_dataset = split_data(torch_data, 0.8)

# load
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

Processed 200 atoms


In [21]:
# ------------------------------- Final Function -------------------------------------------
def param_tuning(param, param_start: int, param_end: int, param_step: int, filelocation):
    # initialize numpy array for param vals
    param_vals = np.arange(start=param_start, end=(param_end+1), step=param_step)

    # initialize default params
    default_readout = 'add'
    default_dipole = False
    default_mean = None
    default_std = None
    default_atomref = None
    default_train_mean = None
    default_train_std = None

    # run model
    epochs = 50
    train_losses = np.zeros(epochs)
    val_losses = np.zeros(epochs)

    for epoch in range(50):
        train_loss = train(bio_model, train_loader)
        val_loss = evaluate(bio_model, val_loader)

        train_losses[epoch] = train_loss
        val_losses[epoch] = val_loss

    # save photo of RMSE graph
    plot_losses(bio_train_losses, bio_val_losses)
    # save graph

    # save combos to df then excel (maybe graph?)

In [ ]:
# ================================== Run Baseline Sweep =================================

